In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time, re

# =========================
# CONFIG
# =========================
BASE = "https://today.thefinancialexpress.com.bd"
ARCHIVE = BASE + "/archive"

START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2025, 12, 31)

HEADERS = {"User-Agent": "Mozilla/5.0"}

# =========================
# HELPERS
# =========================
def extract_title(soup):
    # Try h1
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)

    # og:title fallback
    meta = soup.find("meta", property="og:title")
    if meta and meta.get("content"):
        return meta["content"].strip()

    # title tag fallback
    if soup.title:
        return soup.title.get_text(strip=True)

    return ""

def detect_section(url):
    if "/stock" in url:
        return "stock"
    if "/trade" in url:
        return "trade"
    if "/economy" in url:
        return "economy"
    return "other"

# =========================
# SCRAPER
# =========================
records = []
visited = set()

d = START_DATE
while d <= END_DATE:
    date_str = d.strftime("%Y-%m-%d")
    print("📅 Archive:", date_str)

    # Try both possible params
    urls = [
        f"{ARCHIVE}?date={date_str}",
        f"{ARCHIVE}?dt={date_str}"
    ]

    page_html = None
    used_url = None
    for u in urls:
        r = requests.get(u, headers=HEADERS, timeout=30)
        if r.status_code == 200 and len(r.text) > 2000:
            page_html = r.text
            used_url = u
            break

    if not page_html:
        print("⚠️ Archive page not found")
        d += timedelta(days=1)
        continue

    soup = BeautifulSoup(page_html, "html.parser")

    # Collect article links
    links = set()
    for a in soup.select("a[href]"):
        href = a.get("href", "").strip()

        if href.startswith("//"):
            href = "https:" + href

        if href.startswith("/"):
            full = BASE + href
        elif href.startswith("http"):
            full = href
        else:
            continue

        if "today.thefinancialexpress.com.bd" not in full:
            continue
        if "/archive" in full:
            continue

        # Keep finance-related URLs
        if any(k in full for k in ["/stock", "/trade", "/economy"]):
            links.add(full)

    print(f"   🔗 links found: {len(links)}  ({used_url})")

    for link in links:
        if link in visited:
            continue
        visited.add(link)

        try:
            art = requests.get(link, headers=HEADERS, timeout=30)
            asoup = BeautifulSoup(art.text, "html.parser")

            title = extract_title(asoup)
            if not title:
                continue

            content = " ".join(p.get_text(strip=True) for p in asoup.select("article p"))
            if not content:
                content = " ".join(p.get_text(strip=True) for p in asoup.select("p"))

            section = detect_section(link)

            records.append({
                "paper": "FinancialExpress",
                "section": section,
                "date": date_str,
                "title": title,
                "url": link,
                "content": content
            })

        except Exception as e:
            print("   ❌ Skip:", e)

        time.sleep(0.2)

    d += timedelta(days=1)

df = pd.DataFrame(records)

print("\n✅ TOTAL articles scraped:", len(df))
if not df.empty:
    print(df["section"].value_counts())
df.head()

📅 Archive: 2025-01-01
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-01)
📅 Archive: 2025-01-02
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-02)
📅 Archive: 2025-01-03
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-03)
📅 Archive: 2025-01-04
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-04)
📅 Archive: 2025-01-05
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-05)
📅 Archive: 2025-01-06
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-06)
📅 Archive: 2025-01-07
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-07)
📅 Archive: 2025-01-08
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-08)
📅 Archive: 2025-01-09
   🔗 links found: 6  (https://today.thefinancialexpress.com.bd/archive?date=2025-01-09)
📅 Archive:

,paper,section,date,title,url,content
0,FinancialExpress,trade,2025-01-01,The Financial Express,https://today.thefinancialexpress.com.bd/trade...,"LONDON, Dec 31 (Reuters): Oil prices were on t..."
1,FinancialExpress,trade,2025-01-01,The Financial Express,https://today.thefinancialexpress.com.bd/trade...,Industry experts on Tuesday stressed the urgen...
2,FinancialExpress,stock,2025-01-01,The Financial Express,https://today.thefinancialexpress.com.bd/stock...,The market decides its own course now as the A...
3,FinancialExpress,stock,2025-01-01,The Financial Express,https://today.thefinancialexpress.com.bd/stock...,Economies of scale are helping Bangladesh Stee...
4,FinancialExpress,trade,2025-01-01,The Financial Express,https://today.thefinancialexpress.com.bd/trade...,Thailand's rice prices climbed to their highes...


In [3]:
print("Before dedup:", len(df))
df = df.drop_duplicates(subset=["url"])
print("After dedup:", len(df))

Before dedup: 1098
After dedup: 1098


In [4]:
from google.colab import files

OUT = "/content/financialexpress_archive_finance_2025.csv"
df.to_csv(OUT, index=False, encoding="utf-8-sig")

print("✅ CSV READY")
print("📄 File:", OUT)
print("📊 Rows:", len(df))

files.download(OUT)

✅ CSV READY
📄 File: /content/financialexpress_archive_finance_2025.csv
📊 Rows: 1098


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>